
# OLMo-2 Semantic Induction Head / Relation Index Developmental Sweep

This notebook is the RI counterpart of the organized ICL checkpoint-sweep notebook.

The behavioral sweep found the clearest sampled onset of few-shot pattern discovery between **3B and 5B training tokens**, with format compliance appearing earlier. The purpose of this notebook is to ask whether Semantic Induction Head structure, measured with the **Relation Index (RI)** of Ren et al., changes over a similar training interval.

The notebook:

- uses the frozen tokenizer-safe AGENDA stream created earlier;
- scans **all 32 × 32 = 1024 OLMo attention heads**;
- uses the paper's relaxed developmental QK gate as the primary metric:
  `source position == attention argmax`;
- computes the original `tau = 2.2` dominance gate simultaneously as a sensitivity analysis;
- uses forward AGENDA head→tail relations as the primary direction; reverse relations can be enabled as a secondary sensitivity analysis;
- uses the same raw-embedding OV decomposition as the existing project Stage-1 implementation;
- saves resumable chunk-level aggregates to Google Drive;
- traces fixed final-checkpoint high-RI heads backward through training;
- keeps QK-pass frequency and null-token scores separate from conditional RI.

No causal conclusion should be drawn from this sweep alone. The goal is to compare **developmental timing** of RI structure with the independently measured ICL trajectory.


# 1. Setup


In [ ]:

!pip -q install transformers huggingface_hub pandas numpy scipy tqdm accelerate


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

import gc
import hashlib
import json
import math
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from huggingface_hub import list_repo_refs
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:

# ------------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/NLP_Project/olmo_sih_dynamics")
DATA_DIR = ROOT / "data"
ICL_RESULTS_DIR = ROOT / "results" / "icl"
RI_RESULTS_DIR = ROOT / "results" / "ri"
RUN_DIR = RI_RESULTS_DIR / "developmental_v1"

MODEL_NAME = "allenai/OLMo-2-1124-7B"
RI_SAFE_PATH = DATA_DIR / "ri_agenda_olmo_safe.jsonl"

RUN_DIR.mkdir(parents=True, exist_ok=True)

# Primary developmental RI uses only the argmax requirement, matching
# Ren et al.'s checkpoint analysis. tau=2.2 is recorded as sensitivity.
STRICT_TAU = 2.2
DIRECTIONS = ["forward"]

# Balanced deterministic AGENDA subset.
# "main" uses up to 100 tokenizer-safe triplets per relation.
# Set to None to use every safe row.
ROWS_PER_RELATION = 100
SUBSET_SEED = "olmo-ri-developmental-v1"

# Chunking makes each checkpoint resumable after a Colab disconnect.
TEXT_GROUPS_PER_CHUNK = 20

# Safe notebook-testing mode. Change to True only when you want to
# download checkpoints and compute missing RI chunks.
RUN_NEW_CHECKPOINTS = True

# Final-checkpoint fixed-head trace.
TOP_K_HEADS = 15
MIN_SCORED_HITS_FOR_TOPK = 10

MODEL_CACHE_DIR = Path("/content/olmo_ri_checkpoint_cache")
XET_CACHE_DIR = Path("/root/.cache/huggingface/xet")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RELATIONS = {
    "PART-OF",
    "COMPARE",
    "USED-FOR",
    "FEATURE-OF",
    "HYPONYM-OF",
    "EVALUATE-FOR",
    "CONJUNCTION",
}

print("Project root:", ROOT)
print("RI run directory:", RUN_DIR)
print("Model:", MODEL_NAME)
print("Run unfinished checkpoints:", RUN_NEW_CHECKPOINTS)


## 1.1 Shared file utilities


In [ ]:

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def stable_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


## 1.2 Load the OLMo tokenizer


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)


# 2. Load and Audit the Frozen AGENDA RI Dataset


The audit distinguishes **fatal integrity failures** from **RI-ineligible self-relations**. A row whose AGENDA head and tail are the same entity collapses the RI source and target to the same token. Such rows are preserved in the frozen source file and written to an exclusion manifest, but they are not sampled for the RI experiment.


In [ ]:

ri_rows = load_jsonl(RI_SAFE_PATH)
ri_hash = sha256_file(RI_SAFE_PATH)

print("RI file:", RI_SAFE_PATH)
print("Rows:", len(ri_rows))
print("SHA256:", ri_hash)
print("\nRelation counts:")
print(Counter(row["relation"] for row in ri_rows))


In [ ]:
REQUIRED_FIELDS = {
    "id",
    "sample_id",
    "relation",
    "text",
    "head_entity",
    "tail_entity",
    "head_letter",
    "tail_letter",
    "head_token_positions",
    "tail_token_positions",
    "token_safe",
    "head_token_id",
    "tail_token_id",
}

def as_position_list(value):
    if isinstance(value, int):
        return [value]
    return [int(x) for x in value]


def audit_ri_row(row):
    """
    Separate true data-integrity failures from RI-ineligible self-relations.

    A self-relation is not useful for the head->tail RI test because source
    and target collapse to the same token. We preserve these rows in the
    frozen source file, record them explicitly, and exclude them only from
    the RI assessment.
    """
    errors = []
    exclusions = []

    missing = REQUIRED_FIELDS - set(row)
    if missing:
        return {
            "id": row.get("id"),
            "errors": [f"missing:{sorted(missing)}"],
            "exclusions": [],
            "n_tokens": np.nan,
        }

    if not row["token_safe"]:
        errors.append("not_token_safe")

    if row["relation"] not in EXPECTED_RELATIONS:
        errors.append("unexpected_relation")

    ids = tokenizer.encode(row["text"], add_special_tokens=False)
    head_positions = as_position_list(row["head_token_positions"])
    tail_positions = as_position_list(row["tail_token_positions"])

    if len(head_positions) != 1:
        errors.append(f"head_positions={head_positions}")

    if len(tail_positions) != 1:
        errors.append(f"tail_positions={tail_positions}")

    if len(head_positions) == 1:
        p = head_positions[0]
        if not (0 <= p < len(ids)):
            errors.append("head_position_out_of_range")
        elif ids[p] != int(row["head_token_id"]):
            errors.append("head_token_id_mismatch")

    if len(tail_positions) == 1:
        p = tail_positions[0]
        if not (0 <= p < len(ids)):
            errors.append("tail_position_out_of_range")
        elif ids[p] != int(row["tail_token_id"]):
            errors.append("tail_token_id_mismatch")

    # Same-position source/target rows are only safe to treat as genuine
    # self-relations if all entity/token metadata also agree.
    if (
        len(head_positions) == 1
        and len(tail_positions) == 1
        and head_positions[0] == tail_positions[0]
    ):
        same_token_id = int(row["head_token_id"]) == int(row["tail_token_id"])
        same_letter = row["head_letter"] == row["tail_letter"]
        same_entity = row["head_entity"] == row["tail_entity"]

        if same_token_id and same_letter and same_entity:
            exclusions.append("self_relation_same_source_and_target")
        else:
            errors.append("inconsistent_head_tail_position_collision")

    return {
        "id": row["id"],
        "errors": errors,
        "exclusions": exclusions,
        "n_tokens": len(ids),
    }


if len({row["id"] for row in ri_rows}) != len(ri_rows):
    raise AssertionError("Duplicate RI row IDs found.")

audit_results = [audit_ri_row(row) for row in ri_rows]

fatal_rows = [result for result in audit_results if result["errors"]]
excluded_results = [result for result in audit_results if result["exclusions"]]
excluded_ids = {result["id"] for result in excluded_results}

ri_eligible_rows = [
    row for row in ri_rows
    if row["id"] not in excluded_ids
]

print("Rows checked:", len(audit_results))
print("Fatal integrity problems:", len(fatal_rows))
print("RI-ineligible self-relations:", len(excluded_results))
print("Eligible RI rows:", len(ri_eligible_rows))

if fatal_rows:
    print("\nFatal problems:")
    display(pd.DataFrame(fatal_rows[:30]))
    raise AssertionError("RI dataset integrity audit failed.")

if excluded_results:
    print(
        "\nThese rows are preserved in the frozen source JSONL but excluded "
        "from RI because source and target are the same entity/token:"
    )
    display(pd.DataFrame(excluded_results[:30]))

excluded_rows = [
    {
        **row,
        "ri_exclusion_reason": "self_relation_same_source_and_target",
    }
    for row in ri_rows
    if row["id"] in excluded_ids
]

excluded_path = RUN_DIR / "ri_excluded_self_relations.jsonl"
write_jsonl(excluded_rows, excluded_path)

print("\nALL NON-SELF RI DATASET INTEGRITY CHECKS PASSED")
print("Excluded-row manifest:", excluded_path)
print(
    "Token-length range:",
    min(x["n_tokens"] for x in audit_results if np.isfinite(x["n_tokens"])),
    "to",
    max(x["n_tokens"] for x in audit_results if np.isfinite(x["n_tokens"])),
)

print("\nExcluded self-relations by relation:")
if excluded_rows:
    print(
        pd.Series(Counter(row["relation"] for row in excluded_rows))
        .sort_index()
        .to_string()
    )
else:
    print("None")


### Methodological note: function-word preprocessing

Ren et al. additionally remove frequent function words identified with spaCy after replacing AGENDA entities with single letters. The frozen `ri_agenda_olmo_safe.jsonl` stream used in this project **does not include that exact spaCy function-word-removal step**; it instead uses relevant-sentence extraction, safe-letter entity replacement, unambiguous single occurrences, and OLMo tokenizer-safety checks.

This notebook deliberately does **not** silently change the already-frozen dataset. The difference is recorded as a methodological deviation. If the main developmental RI result is promising, a later sensitivity run can construct a separately named stopword-filtered stream rather than modifying these primary inputs in place.



# 3. Freeze the RI Assessment Subset

The source paper uses the AGENDA test set. Our stream has already been transformed so that the source and target entities are represented by tokenizer-safe single-letter occurrences.

For the developmental sweep we use a **deterministic, relation-balanced subset**. This avoids allowing a frequent relation to dominate the aggregate curve and makes every checkpoint operate on exactly the same triplets.

Set `ROWS_PER_RELATION = None` above if you later want a full-dataset confirmation run.


In [ ]:
def deterministic_relation_subset(rows, n_per_relation, seed):
    grouped = defaultdict(list)

    for row in rows:
        grouped[row["relation"]].append(row)

    available = {rel: len(group) for rel, group in grouped.items()}
    if set(available) != EXPECTED_RELATIONS:
        raise AssertionError(
            f"Unexpected relation set: {sorted(available)}"
        )

    if n_per_relation is None:
        chosen_n = min(available.values())
    else:
        chosen_n = min(int(n_per_relation), min(available.values()))

    selected = []

    for relation in sorted(grouped):
        ranked = sorted(
            grouped[relation],
            key=lambda row: stable_hash(f"{seed}|{relation}|{row['id']}"),
        )
        selected.extend(ranked[:chosen_n])

    return selected, chosen_n, available


# IMPORTANT: sample only from rows that passed integrity checks and are
# non-self relations. The original frozen JSONL itself is never overwritten.
ri_assessment, n_per_relation, available_counts = deterministic_relation_subset(
    ri_eligible_rows,
    ROWS_PER_RELATION,
    SUBSET_SEED,
)

assessment_manifest_path = RUN_DIR / "ri_assessment_manifest.jsonl"
write_jsonl(ri_assessment, assessment_manifest_path)

print("Available eligible rows by relation:")
print(pd.Series(available_counts).sort_index().to_string())
print("\nFrozen rows per relation:", n_per_relation)
print("Total RI triplets:", len(ri_assessment))
print("Excluded self-relations before sampling:", len(excluded_rows))
print("Assessment manifest:", assessment_manifest_path)

display(
    pd.DataFrame(ri_assessment)
    .groupby("relation")
    .size()
    .rename("n")
    .reset_index()
)

## 3.1 Group identical text so one model forward can score multiple triplets


In [ ]:

def build_text_groups(rows):
    grouped = defaultdict(list)

    for row in rows:
        grouped[row["text"]].append(row)

    groups = []

    for text, group_rows in grouped.items():
        groups.append({
            "group_id": stable_hash(text)[:16],
            "text": text,
            "rows": sorted(group_rows, key=lambda row: row["id"]),
        })

    return sorted(groups, key=lambda group: group["group_id"])


text_groups = build_text_groups(ri_assessment)

chunks = [
    text_groups[i:i + TEXT_GROUPS_PER_CHUNK]
    for i in range(0, len(text_groups), TEXT_GROUPS_PER_CHUNK)
]

chunk_manifest = []
for chunk_index, chunk in enumerate(chunks):
    chunk_manifest.append({
        "chunk_index": chunk_index,
        "n_text_groups": len(chunk),
        "n_triplets": sum(len(group["rows"]) for group in chunk),
        "first_group_id": chunk[0]["group_id"],
        "last_group_id": chunk[-1]["group_id"],
    })

chunk_manifest_df = pd.DataFrame(chunk_manifest)
chunk_manifest_df.to_csv(RUN_DIR / "ri_chunk_manifest.csv", index=False)

print("Unique texts:", len(text_groups))
print("Chunks:", len(chunks))
display(chunk_manifest_df)


# 4. Discover OLMo Stage-1 Checkpoints


In [ ]:

refs = list_repo_refs(MODEL_NAME)

stage1_revisions = [
    branch.name
    for branch in refs.branches
    if branch.name.startswith("stage1-step")
]

CHECKPOINT_RE = re.compile(r"stage1-step(\d+)-tokens(\d+)B")

checkpoint_info = []

for revision in stage1_revisions:
    match = CHECKPOINT_RE.fullmatch(revision)
    if match is None:
        continue

    checkpoint_info.append({
        "revision": revision,
        "step": int(match.group(1)),
        "tokens_B": int(match.group(2)),
    })

checkpoint_info = sorted(
    checkpoint_info,
    key=lambda x: (x["step"], x["tokens_B"]),
)

print("Parsed Stage-1 checkpoints:", len(checkpoint_info))
print("Earliest:", checkpoint_info[0])
print("Latest:", checkpoint_info[-1])



## 4.1 Checkpoint schedule

The ICL sweep brackets the clearest sampled pattern-discovery transition between 3B and 5B tokens. We therefore sample Stage-1 much more densely in that interval, while retaining later reference points.

The default RI schedule is:

- 1B: clear pre-transition reference
- 3B / step 600: sampled behavioral pre-transition checkpoint
- 3B / step 700: denser early point
- 4B / step 850
- 4B / step 900
- 5B / step 1000: first sampled behavioral checkpoint with clear ICL
- 9B: early post-transition
- 13B: strong multiclass regime
- 38B: strong later ICL regime / nine-class peak in the sampled behavioral sweep
- 3896B: final Stage-1 reference

If the behavioral sweep is refined at the 3B/4B checkpoints, these exact revisions can be compared directly.


In [ ]:

TARGET_STEPS = [
    150,      # 1B
    600,      # 3B
    700,      # 3B
    850,      # 4B
    900,      # 4B
    1000,     # 5B
    2000,     # 9B
    3000,     # 13B
    9000,     # 38B
    928646,   # 3896B
]

by_step = {cp["step"]: cp for cp in checkpoint_info}
missing_steps = [step for step in TARGET_STEPS if step not in by_step]

if missing_steps:
    raise RuntimeError(f"Requested Stage-1 steps not found: {missing_steps}")

selected_checkpoints = [by_step[step] for step in TARGET_STEPS]
checkpoint_manifest = pd.DataFrame(selected_checkpoints)
checkpoint_manifest.to_csv(RUN_DIR / "selected_checkpoint_manifest.csv", index=False)

display(checkpoint_manifest)


## 4.2 Behavioral landmarks for reference only


In [ ]:

icl_landmarks_path = ICL_RESULTS_DIR / "analysis" / "icl_transition_landmarks.csv"
icl_mean_path = ICL_RESULTS_DIR / "analysis" / "mean_20shot_across_tasks.csv"

if icl_landmarks_path.exists():
    print("ICL transition landmarks:")
    display(pd.read_csv(icl_landmarks_path))
else:
    print("ICL landmark file not found; RI sweep can still run.")

if icl_mean_path.exists():
    print("\nMean 20-shot behavioral trajectory:")
    display(pd.read_csv(icl_mean_path))



# 5. Relation Index Definition and OLMo Adaptation

For a triplet \(T=(t_s,\mathrm{relation},t_o)\) and current position \(j\geq\max(s,o)\):

1. **QK gate — primary developmental definition.**  
   Keep a head when the actual OLMo attention distribution at position \(j\) has its causal argmax at source position \(s\).

2. **Strict sensitivity gate.**  
   Also record whether
   \[
   A^h_{j,s}/\max_{k\ne s}A^h_{j,k} > 2.2.
   \]
   This is *not* the primary developmental filter because Ren et al. explicitly drop this requirement for early checkpoints.

3. **OV score.**  
   Starting from the raw embedding \(x_j\), project through the individual head's \(W_VW_O\) contribution and then through the unembedding. For every unique visible context token, mean-center the output probabilities and clamp negative values to zero. RI is the target token's positive mass divided by total positive mass.

### Exact context-only optimization

The implementation below does **not** softmax over the entire 100k-token vocabulary. This does not approximate the RI ratio.

If \(p_i=e^{z_i}/Z\), then both the centered numerator and denominator contain the same positive factor \(1/Z\), which cancels. We therefore need logits only for the unique visible context-token IDs. This makes the 1024-head sweep substantially cheaper while preserving Equation (3).

The QK side uses the model's actual eager attention probabilities, so OLMo-specific normalization and RoPE are respected.


In [ ]:

def context_centered_share(logits_by_head, visible_indices, target_index):
    """
    logits_by_head: [H, U] logits for U unique context-token IDs.
    visible_indices: global context-token indices visible by current position j.
    target_index: global context-token index of target ID.

    Returns [H] RI shares. NaN means the positive centered denominator was zero.
    """
    visible_indices = torch.as_tensor(visible_indices, dtype=torch.long)
    visible_logits = logits_by_head[:, visible_indices]

    # Any common multiplicative factor cancels in the final ratio, so
    # subtracting the visible maximum is only for numerical stability.
    shifted = visible_logits - visible_logits.max(dim=1, keepdim=True).values
    weights = torch.exp(shifted)
    q = torch.clamp(weights - weights.mean(dim=1, keepdim=True), min=0)
    denom = q.sum(dim=1)

    local_target = int(
        (visible_indices == int(target_index)).nonzero(as_tuple=True)[0][0]
    )
    numerator = q[:, local_target]

    return torch.where(
        denom > 0,
        numerator / denom,
        torch.full_like(denom, float("nan")),
    )


# Metric-level unit test: explicit full-vocabulary softmax and the
# context-only cancellation must agree.
torch.manual_seed(0)

for _ in range(20):
    full_logits = torch.randn(3, 50)
    context_ids = torch.tensor([2, 7, 11, 19, 31, 42])
    target_local = 3

    p = torch.softmax(full_logits, dim=-1)[:, context_ids]
    q = torch.clamp(p - p.mean(dim=1, keepdim=True), min=0)
    explicit = q[:, target_local] / q.sum(dim=1)

    optimized = context_centered_share(
        full_logits[:, context_ids],
        np.arange(len(context_ids)),
        target_local,
    )

    if not torch.allclose(explicit, optimized, atol=1e-6, rtol=1e-5, equal_nan=True):
        raise AssertionError("Context-only RI optimization failed equivalence test.")

print("PASS: context-only OV RI computation matches explicit full softmax.")


# 6. RI Accumulator and OV Utilities


In [ ]:

ACCUM_INT_FIELDS = [
    "opportunities",
    "argmax_hits",
    "argmax_scored",
    "strict_hits",
    "strict_scored",
    "zero_denom_argmax",
    "null10_scored",
    "random_scored",
]

ACCUM_FLOAT_FIELDS = [
    "ri_sum",
    "strict_ri_sum",
    "null10_sum",
    "random_sum",
]


def new_accumulator(n_relations, n_directions, n_layers, n_heads):
    shape = (n_relations, n_directions, n_layers, n_heads)

    acc = {
        field: np.zeros(shape, dtype=np.int64)
        for field in ACCUM_INT_FIELDS
    }
    acc.update({
        field: np.zeros(shape, dtype=np.float64)
        for field in ACCUM_FLOAT_FIELDS
    })

    return acc


def add_accumulator(dst, src):
    for field in ACCUM_INT_FIELDS + ACCUM_FLOAT_FIELDS:
        dst[field] += src[field]


def save_accumulator(acc, path):
    np.savez_compressed(path, **acc)


def load_accumulator(path):
    with np.load(path) as data:
        return {field: data[field] for field in data.files}


def single_position(row, prefix):
    positions = as_position_list(row[f"{prefix}_token_positions"])
    if len(positions) != 1:
        raise ValueError(f"{row['id']}: {prefix} positions are not singleton: {positions}")
    return int(positions[0])


def deterministic_random_visible_index(row_id, direction, j, visible_global_indices):
    key = f"{SUBSET_SEED}|{row_id}|{direction}|{j}"
    value = int(stable_hash(key)[:16], 16)
    return int(visible_global_indices[value % len(visible_global_indices)])


In [ ]:

def ov_context_logits_for_layer(model, layer_index, unique_token_ids):
    """
    Compute x_j W_V^h W_O^h W_U only for the unique token IDs present
    in the current text.

    Returns CPU float32 tensor [U_current, H, U_context].
    """
    layer = model.model.layers[layer_index]
    attn = layer.self_attn

    n_heads = model.config.num_attention_heads
    n_kv_heads = model.config.num_key_value_heads
    hidden_size = model.config.hidden_size
    head_dim = hidden_size // n_heads

    if n_kv_heads != n_heads:
        raise RuntimeError(
            "This implementation assumes one V head per attention head. "
            f"Found num_attention_heads={n_heads}, num_key_value_heads={n_kv_heads}."
        )

    if attn.o_proj.bias is not None:
        raise RuntimeError("Unexpected o_proj bias; headwise decomposition needs review.")

    W_E = model.get_input_embeddings().weight
    W_U = model.lm_head.weight

    v_device = attn.v_proj.weight.device
    token_ids_e = torch.tensor(unique_token_ids, device=W_E.device)
    X = W_E[token_ids_e].to(v_device, dtype=attn.v_proj.weight.dtype)

    # [U, H * Dh] -> [U, H, Dh]
    V = attn.v_proj(X).view(len(unique_token_ids), n_heads, head_dim)

    # o_proj.weight: [d_model, H*Dh]
    # Transpose and split its input dimension by head.
    O_by_head = (
        attn.o_proj.weight.T
        .contiguous()
        .view(n_heads, head_dim, hidden_size)
    )

    # Independent output contribution from every head: [U, H, d_model]
    Z = torch.einsum("uhd,hdm->uhm", V, O_by_head)

    out_device = W_U.device
    Z = Z.to(out_device)
    token_ids_u = torch.tensor(unique_token_ids, device=out_device)
    W_U_context = W_U[token_ids_u].to(Z.dtype)

    # Only logits of context-token IDs are needed.
    logits = torch.einsum("uhd,vd->uhv", Z, W_U_context)

    result = logits.float().cpu()

    del X, V, O_by_head, Z, W_U_context, token_ids_e, token_ids_u
    return result


# 7. Process One Text Group


In [ ]:

def process_text_group(
    model,
    tokenizer,
    group,
    relation_to_index,
    direction_to_index,
    acc,
):
    text = group["text"]
    rows = group["rows"]

    ids_l = tokenizer.encode(text, add_special_tokens=False)
    n = len(ids_l)

    # Verify frozen token positions against this runtime's tokenizer.
    for row in rows:
        hp = single_position(row, "head")
        tp = single_position(row, "tail")

        if ids_l[hp] != int(row["head_token_id"]):
            raise AssertionError(f"{row['id']}: head token mismatch at runtime")
        if ids_l[tp] != int(row["tail_token_id"]):
            raise AssertionError(f"{row['id']}: tail token mismatch at runtime")

    input_device = model.get_input_embeddings().weight.device
    input_ids = torch.tensor([ids_l], device=input_device)

    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids,
            use_cache=False,
            output_attentions=True,
            return_dict=True,
        )

    if outputs.attentions is None:
        raise RuntimeError(
            "The model did not return attentions. "
            "Load it with attn_implementation='eager'."
        )

    n_layers = model.config.num_hidden_layers
    n_heads = model.config.num_attention_heads

    if len(outputs.attentions) != n_layers:
        raise AssertionError("Unexpected number of returned attention layers.")

    # Unique context tokens in first-occurrence order.
    first_position = {}
    for position, token_id in enumerate(ids_l):
        first_position.setdefault(int(token_id), position)

    unique_token_ids = list(first_position.keys())
    context_index = {
        token_id: i
        for i, token_id in enumerate(unique_token_ids)
    }
    first_positions = np.array(
        [first_position[token_id] for token_id in unique_token_ids],
        dtype=np.int64,
    )

    tok10_id = int(ids_l[10] if n > 10 else ids_l[-1])
    tok10_context_index = context_index[tok10_id]

    # One specification per triplet × direction.
    specs = []

    for row in rows:
        relation_index = relation_to_index[row["relation"]]
        head_position = single_position(row, "head")
        tail_position = single_position(row, "tail")

        for direction in DIRECTIONS:
            direction_index = direction_to_index[direction]

            if direction == "forward":
                source_position = head_position
                target_position = tail_position
                target_id = int(row["tail_token_id"])
            elif direction == "reverse":
                source_position = tail_position
                target_position = head_position
                target_id = int(row["head_token_id"])
            else:
                raise ValueError(f"Unknown direction: {direction}")

            specs.append({
                "row_id": row["id"],
                "relation_index": relation_index,
                "direction": direction,
                "direction_index": direction_index,
                "source_position": source_position,
                "target_position": target_position,
                "target_context_index": context_index[target_id],
                "jmin": max(source_position, target_position),
            })

    with torch.inference_mode():
        for layer_index in range(n_layers):
            A = outputs.attentions[layer_index][0].float().cpu()  # [H, n, n]

            if A.shape != (n_heads, n, n):
                raise AssertionError(
                    f"Unexpected attention shape at layer {layer_index}: {tuple(A.shape)}"
                )

            ov_logits = ov_context_logits_for_layer(
                model,
                layer_index,
                unique_token_ids,
            )  # [U, H, U]

            for spec in specs:
                r = spec["relation_index"]
                d = spec["direction_index"]
                source_position = spec["source_position"]
                target_context_index = spec["target_context_index"]

                for j in range(spec["jmin"], n):
                    acc["opportunities"][r, d, layer_index, :] += 1

                    attention_row = A[:, j, :j + 1]
                    top1, argmax = attention_row.max(dim=-1)

                    if j >= 1:
                        top2 = torch.topk(attention_row, k=2, dim=-1).values[:, 1]
                    else:
                        top2 = torch.zeros_like(top1)

                    ratio = top1 / (top2 + 1e-12)
                    argmax_hit = argmax.eq(source_position)
                    strict_hit = argmax_hit & ratio.gt(STRICT_TAU)

                    hit_heads = np.flatnonzero(argmax_hit.numpy())
                    strict_heads = np.flatnonzero(strict_hit.numpy())

                    if len(hit_heads) == 0:
                        continue

                    acc["argmax_hits"][r, d, layer_index, hit_heads] += 1
                    if len(strict_heads):
                        acc["strict_hits"][r, d, layer_index, strict_heads] += 1

                    current_context_index = context_index[int(ids_l[j])]
                    visible_global = np.flatnonzero(first_positions <= j)

                    shares = context_centered_share(
                        ov_logits[current_context_index],
                        visible_global,
                        target_context_index,
                    )

                    finite = torch.isfinite(shares).numpy()
                    scored_heads = np.flatnonzero(argmax_hit.numpy() & finite)

                    if len(scored_heads):
                        values = shares.numpy()[scored_heads]
                        acc["argmax_scored"][r, d, layer_index, scored_heads] += 1
                        acc["ri_sum"][r, d, layer_index, scored_heads] += values

                    zero_heads = np.flatnonzero(argmax_hit.numpy() & ~finite)
                    if len(zero_heads):
                        acc["zero_denom_argmax"][r, d, layer_index, zero_heads] += 1

                    strict_scored_heads = np.flatnonzero(
                        strict_hit.numpy() & finite
                    )
                    if len(strict_scored_heads):
                        values = shares.numpy()[strict_scored_heads]
                        acc["strict_scored"][r, d, layer_index, strict_scored_heads] += 1
                        acc["strict_ri_sum"][r, d, layer_index, strict_scored_heads] += values

                    # Fixed token-at-position-10 null, matching the paper/project control.
                    if first_positions[tok10_context_index] <= j:
                        null10 = context_centered_share(
                            ov_logits[current_context_index],
                            visible_global,
                            tok10_context_index,
                        )
                        null10_finite = torch.isfinite(null10).numpy()
                        null10_heads = np.flatnonzero(
                            argmax_hit.numpy() & null10_finite
                        )
                        if len(null10_heads):
                            values = null10.numpy()[null10_heads]
                            acc["null10_scored"][r, d, layer_index, null10_heads] += 1
                            acc["null10_sum"][r, d, layer_index, null10_heads] += values

                    # Same deterministic random visible context token at every checkpoint.
                    random_context_index = deterministic_random_visible_index(
                        spec["row_id"],
                        spec["direction"],
                        j,
                        visible_global,
                    )
                    random_share = context_centered_share(
                        ov_logits[current_context_index],
                        visible_global,
                        random_context_index,
                    )
                    random_finite = torch.isfinite(random_share).numpy()
                    random_heads = np.flatnonzero(
                        argmax_hit.numpy() & random_finite
                    )
                    if len(random_heads):
                        values = random_share.numpy()[random_heads]
                        acc["random_scored"][r, d, layer_index, random_heads] += 1
                        acc["random_sum"][r, d, layer_index, random_heads] += values

            del A, ov_logits

    del outputs, input_ids


# 8. Convert Accumulators to Head-Level Results


In [ ]:

def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)

    out = np.full_like(numerator, np.nan, dtype=np.float64)
    np.divide(numerator, denominator, out=out, where=denominator > 0)
    return out


def accumulator_to_dataframe(acc, checkpoint, relations, directions):
    records = []

    n_layers = acc["opportunities"].shape[2]
    n_heads = acc["opportunities"].shape[3]

    for r, relation in enumerate(relations):
        for d, direction in enumerate(directions):
            for layer in range(n_layers):
                for head in range(n_heads):
                    opportunities = int(acc["opportunities"][r, d, layer, head])
                    argmax_hits = int(acc["argmax_hits"][r, d, layer, head])
                    argmax_scored = int(acc["argmax_scored"][r, d, layer, head])
                    strict_hits = int(acc["strict_hits"][r, d, layer, head])
                    strict_scored = int(acc["strict_scored"][r, d, layer, head])

                    ri_argmax = (
                        acc["ri_sum"][r, d, layer, head] / argmax_scored
                        if argmax_scored > 0 else np.nan
                    )
                    ri_strict = (
                        acc["strict_ri_sum"][r, d, layer, head] / strict_scored
                        if strict_scored > 0 else np.nan
                    )
                    null10_mean = (
                        acc["null10_sum"][r, d, layer, head]
                        / acc["null10_scored"][r, d, layer, head]
                        if acc["null10_scored"][r, d, layer, head] > 0 else np.nan
                    )
                    random_mean = (
                        acc["random_sum"][r, d, layer, head]
                        / acc["random_scored"][r, d, layer, head]
                        if acc["random_scored"][r, d, layer, head] > 0 else np.nan
                    )

                    records.append({
                        "checkpoint": checkpoint["revision"],
                        "step": checkpoint["step"],
                        "tokens_B": checkpoint["tokens_B"],
                        "relation": relation,
                        "direction": direction,
                        "layer": layer,
                        "head": head,
                        "head_name": f"L{layer}H{head}",
                        "opportunities": opportunities,
                        "argmax_hits": argmax_hits,
                        "argmax_scored": argmax_scored,
                        "qk_argmax_frequency": (
                            argmax_hits / opportunities if opportunities else np.nan
                        ),
                        "ri_argmax": ri_argmax,
                        # Defined for every head; non-passing events contribute zero.
                        "ri_per_opportunity": (
                            acc["ri_sum"][r, d, layer, head] / opportunities
                            if opportunities else np.nan
                        ),
                        "strict_hits": strict_hits,
                        "strict_scored": strict_scored,
                        "qk_strict_frequency": (
                            strict_hits / opportunities if opportunities else np.nan
                        ),
                        "ri_strict_tau_2_2": ri_strict,
                        "zero_denom_argmax": int(
                            acc["zero_denom_argmax"][r, d, layer, head]
                        ),
                        "null_tok10_mean": null10_mean,
                        "null_random_mean": random_mean,
                        "ri_minus_null_tok10": (
                            ri_argmax - null10_mean
                            if np.isfinite(ri_argmax) and np.isfinite(null10_mean)
                            else np.nan
                        ),
                        "ri_minus_null_random": (
                            ri_argmax - random_mean
                            if np.isfinite(ri_argmax) and np.isfinite(random_mean)
                            else np.nan
                        ),
                    })

    return pd.DataFrame(records)


# 9. Model Loading and Local-Cache Cleanup


In [ ]:

def preferred_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def print_local_disk_usage(prefix="Disk"):
    usage = shutil.disk_usage("/")
    gb = 1024 ** 3
    print(
        f"{prefix}: {usage.used / gb:.1f} GB used, "
        f"{usage.free / gb:.1f} GB free of {usage.total / gb:.1f} GB"
    )


def clear_checkpoint_disk_cache():
    if MODEL_CACHE_DIR.exists():
        shutil.rmtree(MODEL_CACHE_DIR, ignore_errors=True)

    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if XET_CACHE_DIR.exists():
        shutil.rmtree(XET_CACHE_DIR, ignore_errors=True)


def load_checkpoint_for_ri(revision, min_free_gb=50):
    if not torch.cuda.is_available():
        raise RuntimeError("A GPU runtime is required for the RI sweep.")

    clear_checkpoint_disk_cache()

    free_gb = shutil.disk_usage("/").free / (1024 ** 3)
    if free_gb < min_free_gb:
        raise RuntimeError(
            f"Only {free_gb:.1f} GB local disk is free; "
            f"{min_free_gb} GB is required before checkpoint download."
        )

    print("\n" + "=" * 88)
    print("LOADING RI CHECKPOINT:", revision)
    print_local_disk_usage("Before checkpoint download")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=revision,
        torch_dtype=preferred_dtype(),
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
        cache_dir=str(MODEL_CACHE_DIR),
    )

    model.eval()
    model.config.use_cache = False

    print(
        "Architecture:",
        f"{model.config.num_hidden_layers} layers × "
        f"{model.config.num_attention_heads} heads",
    )
    print("KV heads:", model.config.num_key_value_heads)

    if model.config.num_key_value_heads != model.config.num_attention_heads:
        raise RuntimeError("Unexpected GQA configuration; OV decomposition needs adaptation.")

    return model


def release_model(model):
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    clear_checkpoint_disk_cache()
    print_local_disk_usage("After checkpoint cleanup")


# 10. Resumable Checkpoint Runner


In [ ]:

RELATIONS = sorted(EXPECTED_RELATIONS)
RELATION_TO_INDEX = {relation: i for i, relation in enumerate(RELATIONS)}
DIRECTION_TO_INDEX = {direction: i for i, direction in enumerate(DIRECTIONS)}


def expected_chunk_paths(checkpoint):
    chunk_dir = RUN_DIR / "chunks" / checkpoint["revision"]
    chunk_dir.mkdir(parents=True, exist_ok=True)

    return [
        chunk_dir / f"chunk_{i:04d}.npz"
        for i in range(len(chunks))
    ]


def combine_checkpoint_chunks(checkpoint, n_layers=32, n_heads=32):
    paths = expected_chunk_paths(checkpoint)
    missing = [path for path in paths if not path.exists()]

    if missing:
        raise RuntimeError(
            f"{checkpoint['revision']}: {len(missing)} RI chunks are missing."
        )

    combined = new_accumulator(
        len(RELATIONS),
        len(DIRECTIONS),
        n_layers,
        n_heads,
    )

    for path in paths:
        add_accumulator(combined, load_accumulator(path))

    return combined


def run_checkpoint_resumable(model, checkpoint):
    paths = expected_chunk_paths(checkpoint)
    missing_indices = [
        i for i, path in enumerate(paths)
        if not path.exists()
    ]

    print(
        f"\n{checkpoint['revision']}: "
        f"{len(paths) - len(missing_indices)}/{len(paths)} chunks already complete."
    )

    if missing_indices:
        # Model-side preflight before the first expensive chunk.
        first_group = chunks[missing_indices[0]][0]
        first_ids = tokenizer.encode(first_group["text"], add_special_tokens=False)

        with torch.inference_mode():
            test_ids = torch.tensor(
                [first_ids],
                device=model.get_input_embeddings().weight.device,
            )
            test_output = model(
                input_ids=test_ids,
                output_attentions=True,
                use_cache=False,
                return_dict=True,
            )

        if test_output.attentions is None:
            raise RuntimeError("RI preflight failed: attentions were not returned.")

        print(
            "Preflight attention shape:",
            tuple(test_output.attentions[0].shape),
        )
        del test_output, test_ids

    for chunk_index in tqdm(
        missing_indices,
        desc=f"{checkpoint['tokens_B']}B chunks",
    ):
        chunk_acc = new_accumulator(
            len(RELATIONS),
            len(DIRECTIONS),
            model.config.num_hidden_layers,
            model.config.num_attention_heads,
        )

        for group in tqdm(
            chunks[chunk_index],
            desc=f"chunk {chunk_index:04d}",
            leave=False,
        ):
            process_text_group(
                model=model,
                tokenizer=tokenizer,
                group=group,
                relation_to_index=RELATION_TO_INDEX,
                direction_to_index=DIRECTION_TO_INDEX,
                acc=chunk_acc,
            )

        save_accumulator(chunk_acc, paths[chunk_index])

    combined = combine_checkpoint_chunks(
        checkpoint,
        n_layers=model.config.num_hidden_layers,
        n_heads=model.config.num_attention_heads,
    )

    head_stats = accumulator_to_dataframe(
        combined,
        checkpoint,
        RELATIONS,
        DIRECTIONS,
    )

    head_stats_path = RUN_DIR / f"{checkpoint['revision']}_head_stats.csv"
    head_stats.to_csv(head_stats_path, index=False)

    print("Saved head statistics:", head_stats_path)

    # Compact sanity summary.
    sanity = (
        head_stats.groupby(["relation", "direction"])
        .agg(
            max_ri=("ri_argmax", "max"),
            median_ri=("ri_argmax", "median"),
            mean_qk_frequency=("qk_argmax_frequency", "mean"),
            heads_with_scored_hits=("argmax_scored", lambda s: int((s > 0).sum())),
        )
        .reset_index()
    )
    print(sanity.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return head_stats



# 11. Run or Recover the RI Sweep

With `RUN_NEW_CHECKPOINTS = False`, this cell only recognizes already-complete checkpoint outputs and never loads model weights.

With `RUN_NEW_CHECKPOINTS = True`, it downloads a checkpoint only when one or more chunk files are missing. Completed chunks are never recomputed.

A disconnect during a chunk loses only that chunk; all earlier chunks remain on Drive.


In [ ]:
run_config = {
    "model": MODEL_NAME,
    "ri_data_sha256": ri_hash,
    "eligible_ids_sha256": stable_hash(
        "\n".join(sorted(row["id"] for row in ri_eligible_rows))
    ),
    "excluded_self_relation_count": len(excluded_rows),
    "assessment_ids_sha256": stable_hash(
        "\n".join(sorted(row["id"] for row in ri_assessment))
    ),
    "rows_per_relation": n_per_relation,
    "relations": RELATIONS,
    "directions": DIRECTIONS,
    "strict_tau": STRICT_TAU,
    "text_groups_per_chunk": TEXT_GROUPS_PER_CHUNK,
    "target_steps": TARGET_STEPS,
    "primary_qk_gate": "source_is_attention_argmax",
    "ov_input": "raw_token_embedding",
    "ov_normalization": "positive_mean_centered_visible_unique_context_tokens",
    "self_relation_policy": (
        "preserve_in_frozen_source_but_exclude_from_RI_when_"
        "head_entity_equals_tail_entity_and_positions_coincide"
    ),
}

config_path = RUN_DIR / "run_config.json"

if config_path.exists():
    old_config = json.loads(config_path.read_text())

    if old_config != run_config:
        raise RuntimeError(
            "Existing RI run_config.json does not match this notebook configuration. "
            "Use a new RUN_DIR or restore the original configuration before resuming."
        )
else:
    config_path.write_text(json.dumps(run_config, indent=2))

print("Run config:", config_path)

In [ ]:

completed_head_stats = []

checkpoint_progress = tqdm(selected_checkpoints, desc="RI CHECKPOINTS")

for checkpoint in checkpoint_progress:
    revision = checkpoint["revision"]
    checkpoint_progress.set_postfix(step=checkpoint["step"], tokens=f"{checkpoint['tokens_B']}B")

    head_stats_path = RUN_DIR / f"{revision}_head_stats.csv"
    paths = expected_chunk_paths(checkpoint)
    chunks_complete = all(path.exists() for path in paths)

    if head_stats_path.exists() and chunks_complete:
        print(f"\nALREADY COMPLETE: {revision} — no model load.")
        completed_head_stats.append(pd.read_csv(head_stats_path))
        continue

    if chunks_complete and not head_stats_path.exists():
        print(f"\nREBUILDING AGGREGATE ONLY: {revision}")
        combined = combine_checkpoint_chunks(checkpoint)
        stats = accumulator_to_dataframe(
            combined,
            checkpoint,
            RELATIONS,
            DIRECTIONS,
        )
        stats.to_csv(head_stats_path, index=False)
        completed_head_stats.append(stats)
        continue

    if not RUN_NEW_CHECKPOINTS:
        done = sum(path.exists() for path in paths)
        print(
            f"\nNOT RUN: {revision} — "
            f"{done}/{len(paths)} chunks currently saved."
        )
        continue

    model = None

    try:
        model = load_checkpoint_for_ri(revision)
        stats = run_checkpoint_resumable(model, checkpoint)
        completed_head_stats.append(stats)

    finally:
        if model is not None:
            release_model(model)
            model = None
        else:
            clear_checkpoint_disk_cache()


# 12. Combine Completed Checkpoints


In [ ]:

# Reload from disk so this cell also works after a fresh runtime.
frames = []

for checkpoint in selected_checkpoints:
    path = RUN_DIR / f"{checkpoint['revision']}_head_stats.csv"
    if path.exists():
        frames.append(pd.read_csv(path))

if not frames:
    print(
        "No completed RI checkpoint aggregates exist yet. "
        "Set RUN_NEW_CHECKPOINTS = True when you are ready to start the sweep."
    )
    ri_trajectory = pd.DataFrame()
else:
    ri_trajectory = (
        pd.concat(frames, ignore_index=True)
        .sort_values(["step", "relation", "direction", "layer", "head"])
        .reset_index(drop=True)
    )

    trajectory_path = RUN_DIR / "ri_head_trajectory.csv"
    ri_trajectory.to_csv(trajectory_path, index=False)

    print("Completed checkpoints:", ri_trajectory["checkpoint"].nunique())
    print("Trajectory rows:", len(ri_trajectory))
    print("Saved:", trajectory_path)


# 13. Minimal Sweep Summary and Fixed Final-Head Trace


In [ ]:

if ri_trajectory.empty:
    print("Run at least one RI checkpoint before this section.")
else:
    checkpoint_summary = (
        ri_trajectory.groupby(["checkpoint", "step", "tokens_B", "relation", "direction"])
        .apply(
            lambda g: pd.Series({
                "max_ri_argmax": g["ri_argmax"].max(),
                "mean_ri_per_opportunity": g["ri_per_opportunity"].mean(),
                "mean_qk_argmax_frequency": g["qk_argmax_frequency"].mean(),
                "n_heads_with_hits": int((g["argmax_scored"] > 0).sum()),
                "top15_ri_mean": g.loc[
                    g["argmax_scored"] >= MIN_SCORED_HITS_FOR_TOPK,
                    "ri_argmax",
                ].nlargest(TOP_K_HEADS).mean(),
            })
        )
        .reset_index()
        .sort_values(["step", "relation", "direction"])
    )

    checkpoint_summary.to_csv(
        RUN_DIR / "ri_checkpoint_summary.csv",
        index=False,
    )

    print(
        checkpoint_summary.to_string(
            index=False,
            float_format=lambda x: f"{x:.5f}",
        )
    )


In [ ]:

if not ri_trajectory.empty:
    final_step = max(
        step for step in TARGET_STEPS
        if step in set(ri_trajectory["step"])
    )
    final_df = ri_trajectory.loc[ri_trajectory["step"] == final_step].copy()

    final_top_rows = []

    for (relation, direction), group in final_df.groupby(["relation", "direction"]):
        eligible = group.loc[
            group["argmax_scored"] >= MIN_SCORED_HITS_FOR_TOPK
        ].copy()

        top = eligible.nlargest(TOP_K_HEADS, "ri_argmax")

        for rank, row in enumerate(top.itertuples(index=False), start=1):
            final_top_rows.append({
                "relation": relation,
                "direction": direction,
                "rank": rank,
                "layer": row.layer,
                "head": row.head,
                "head_name": row.head_name,
                "final_ri_argmax": row.ri_argmax,
                "final_argmax_scored": row.argmax_scored,
            })

    final_top_heads = pd.DataFrame(final_top_rows)
    final_top_heads.to_csv(RUN_DIR / "final_top_heads.csv", index=False)

    fixed_trace = ri_trajectory.merge(
        final_top_heads[
            ["relation", "direction", "rank", "layer", "head"]
        ],
        on=["relation", "direction", "layer", "head"],
        how="inner",
    ).sort_values(["relation", "direction", "rank", "step"])

    fixed_trace.to_csv(
        RUN_DIR / "final_top_heads_backward_trace.csv",
        index=False,
    )

    print("Final reference step:", final_step)
    print("Selected fixed heads:", len(final_top_heads))
    display(final_top_heads.head(30))


## 13.1 Head-set stability relative to the final checkpoint


In [ ]:

if not ri_trajectory.empty and "final_top_heads" in globals():
    stability_rows = []

    final_sets = {
        (relation, direction): set(zip(group["layer"], group["head"]))
        for (relation, direction), group
        in final_top_heads.groupby(["relation", "direction"])
    }

    for (checkpoint, step, tokens_B, relation, direction), group in ri_trajectory.groupby(
        ["checkpoint", "step", "tokens_B", "relation", "direction"]
    ):
        eligible = group.loc[
            group["argmax_scored"] >= MIN_SCORED_HITS_FOR_TOPK
        ].copy()

        current_top = eligible.nlargest(TOP_K_HEADS, "ri_argmax")
        current_set = set(zip(current_top["layer"], current_top["head"]))
        final_set = final_sets.get((relation, direction), set())

        union = current_set | final_set
        jaccard = len(current_set & final_set) / len(union) if union else np.nan

        final_group = (
            ri_trajectory.loc[
                (ri_trajectory["step"] == final_step)
                & (ri_trajectory["relation"] == relation)
                & (ri_trajectory["direction"] == direction),
                ["layer", "head", "ri_per_opportunity"],
            ]
            .rename(columns={"ri_per_opportunity": "final_ri_per_opportunity"})
        )

        aligned = group[
            ["layer", "head", "ri_per_opportunity"]
        ].merge(final_group, on=["layer", "head"], how="inner")

        rho = spearmanr(
            aligned["ri_per_opportunity"],
            aligned["final_ri_per_opportunity"],
        ).statistic

        stability_rows.append({
            "checkpoint": checkpoint,
            "step": step,
            "tokens_B": tokens_B,
            "relation": relation,
            "direction": direction,
            "topk_jaccard_vs_final": jaccard,
            "all_head_spearman_vs_final": rho,
            "eligible_heads": len(eligible),
        })

    head_stability = (
        pd.DataFrame(stability_rows)
        .sort_values(["relation", "direction", "step"])
    )

    head_stability.to_csv(
        RUN_DIR / "head_stability_vs_final.csv",
        index=False,
    )

    print(
        head_stability.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )



# 14. Outputs and Next Step

The sweep writes all results under:

`NLP_Project/olmo_sih_dynamics/results/ri/developmental_v1/`

Important outputs:

- `run_config.json` — frozen methodological configuration
- `ri_assessment_manifest.jsonl` — exact AGENDA triplets used
- `ri_chunk_manifest.csv` — resumable chunk layout
- `selected_checkpoint_manifest.csv` — exact OLMo revisions
- `chunks/<revision>/chunk_XXXX.npz` — persistent resumable aggregates
- `<revision>_head_stats.csv` — all 1024 heads × relation × direction
- `ri_head_trajectory.csv` — combined head-level developmental trajectory
- `ri_checkpoint_summary.csv` — compact per-relation checkpoint summaries
- `final_top_heads.csv` — fixed high-RI heads at the final available reference checkpoint
- `final_top_heads_backward_trace.csv` — those fixed coordinates traced backward
- `head_stability_vs_final.csv` — top-k Jaccard and all-head Spearman stability

The **analysis should be done in a separate notebook**, just as with the ICL sweep. That notebook should compare RI timing against the behavioral 3B–5B transition without treating correlation as causal evidence.
